[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/07_%E3%83%9B%E3%83%93%E3%83%BC_%E9%99%B6%E8%8A%B83D/03_SolidPython%E3%81%A7%E5%99%A8%E3%82%92%E3%81%A4%E3%81%8F%E3%82%8B.ipynb)

> Colabで実行可。最初に「① セットアップ」セルを実行（Colabはデータを自動生成、ローカルは何もしない）。

In [ ]:
# === ① セットアップ（最初に実行）: Colabでは solidpython2 を入れます ===
try:
    import solid2
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'solidpython2'])
print('準備OK')

# ホビー-03. SolidPython で器を「コードで」つくる

**SolidPython** は Python のコードがそのまま3Dモデルになるライブラリ。
書いたコードは **OpenSCAD** という形式(`.scad`)になり、無料ソフトで見られます。

**このノートは Jupyter でそのまま実行できます！**（`.scad` ファイルが書き出されます）

> プログラミングと3Dが完全につながる、いちばん「理系」な作り方です。

> **このツールはこんな時に**：寸法を数値で扱い、**サイズ違いを一括生成**（入れ子の鉢セット・食器ライン）。収縮率を逆算した**石こう型の原型**づくり。

## 0. 準備

**ライブラリのインストール**（このセルを実行）

In [ ]:
# uv環境なら: ターミナルで  uv pip install solidpython2
# このノート内で入れるなら↓のコメントを外す
# %pip install solidpython2
from solid2 import polygon, rotate_extrude, cylinder, scad_render_to_file   # 使う機能を読み込む
print('準備OK')

`.scad` を**見る**には [OpenSCAD](https://openscad.org/)（無料）をインストールし、
書き出したファイルを開きます。OpenSCADから `.stl` も書き出せます。

### 陶芸ならではの注意：収縮率

粘土は乾燥・焼成で **10〜15%ちぢみます**。焼き上がりで口径120mmにしたいなら、
モデルは `120 ÷ (1 - 0.13) ≈ 138mm` で作ります。これは比の良い練習にもなります。

```python
finish = 120          # 焼き上がりで欲しい口径(mm)
shrink = 0.13         # 収縮率13%
model = finish / (1 - shrink)
print(round(model, 1), 'mm で設計する')
```

## 1. 断面 → 回転体（rotate_extrude）

考え方はFreeCAD・Blenderと同じ「回転体」。
2Dの輪郭 `polygon` を `rotate_extrude` で360°回すと器になります。
点は `(半径, 高さ)`。

In [ ]:
# 茶碗の断面（半径, 高さ）— 内側と外側を一周する閉じた輪郭
profile = [
    (0,  0), (45, 0), (45, 6),    # 底
    (52, 68), (60, 70),           # 外壁→口の外
    (54, 70), (47, 66),           # 口の内（肉厚）
    (39, 8), (0, 8), (0, 0),      # 内壁→内側の底
]

chawan = rotate_extrude(angle=360, _fn=120)(polygon(points=profile))   # 断面を360°回して器に
chawan.save_as_scad('chawan.scad')        # OpenSCAD形式(.scad)で保存
print('chawan.scad を書き出しました。OpenSCADで開いてね')

`_fn=120` は「何角形で円を近似するか」。大きいほどなめらか（その分重い）。

## 2. 関数にして、いろんな器を量産する

プログラミングの強み＝**パラメータを変えるだけで何個でも作れる**こと。
器の「口径・高さ・肉厚」を引数にした関数を作ります。

In [ ]:
# 口径・高さ・肉厚を指定して器を作る関数
def make_bowl(diameter=120, height=70, thickness=6, foot=33, fn=120):
    r = diameter / 2                       # 外側の半径
    inner_r = r - thickness                # 内側の半径
    profile = [                            # 断面の輪郭（半径, 高さ）を組み立てる
        (0, 0), (r*0.75, 0), (r*0.75, thickness),
        (r*0.9, height-2), (r, height),
        (r-3, height), (inner_r*0.95, height-4),
        (inner_r*0.6, thickness+2), (0, thickness+2), (0, 0),
    ]
    bowl = rotate_extrude(angle=360, _fn=fn)(polygon(points=profile))   # 回転体にする
    return bowl                            # 器を返す

# 焼き上がりサイズから収縮を逆算して設計
shrink = 0.13                              # 収縮率13%
yunomi = make_bowl(diameter=80/(1-shrink), height=90/(1-shrink), thickness=5)   # 湯のみ
yunomi.save_as_scad('yunomi.scad')         # 保存

chawan = make_bowl(diameter=120/(1-shrink), height=70/(1-shrink), thickness=6)  # 茶碗
chawan.save_as_scad('chawan2.scad')        # 保存
print('湯のみ・茶碗を書き出しました')

## 3. 高台をくり抜く（引き算）

SolidPythonでは図形どうしを `+`（合体）`-`（くり抜き）で組み合わせられます。

In [ ]:
# 器の底に高台（こうだい）をくり抜く関数
def add_foot(bowl, foot_d=66, foot_h=6, fn=120):
    cutter = cylinder(d=foot_d, h=foot_h, _fn=fn)   # くり抜く円柱
    return bowl - cutter   # 底をへこませて高台に

chawan_kodai = add_foot(make_bowl())       # 高台つきの茶碗を作る
chawan_kodai.save_as_scad('chawan_kodai.scad')   # 保存
print('高台つきの茶碗を書き出しました')

## 4. 書き出したファイルを確認

In [ ]:
import glob, os                            # ファイル一覧と情報を扱う
# 書き出した .scad ファイルを一覧表示
for f in sorted(glob.glob('*.scad')):
    print(f, f'({os.path.getsize(f)} bytes)')   # ファイル名とサイズ

---
## チャレンジ課題

**課題1.** `make_bowl` を使って「小皿（口径150・高さ25・肉厚5）」を作ろう。

**課題2.** for文で、口径を 80→160mm まで20mm刻みで変えた器を5個、
`bowl_80.scad` … のように連番で書き出そう。

**課題3.**（発展）`profile` を工夫して、口が花びらのように波打つ器に挑戦（ヒント：
`rotate_extrude` の前に複数の山を作るのは難しいので、まず口径を変えた輪で重ねてみる）。

**課題4.**（統計とつなぐ）課題2で作った5個の器の「容積のおよそ」を円柱で近似計算し、
`02_統計` で習った平均・標準偏差を求めてみよう。

In [ ]:
# 課題1


In [ ]:
# 課題2
for d in range(80, 161, 20):   # 口径80〜160mmを20mm刻みで
    pass  # ここに書く

> **解答例は別ページにまとめています**（ネタバレ防止）。
> 自分で解いてから **[03_SolidPythonで器をつくる の解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/07_%E3%83%9B%E3%83%93%E3%83%BC_%E9%99%B6%E8%8A%B83D/03_SolidPython%E3%81%A7%E5%99%A8%E3%82%92%E3%81%A4%E3%81%8F%E3%82%8B.md)**

**ホビー編クリア！** FreeCAD（精密）・Blender（自由曲面）・SolidPython（コード）の
3つの作り方を体験しました。同じ「回転体」でも道具で得意分野が違うのが面白いところです。

## 容量(ml)を計算する

器の内側の輪郭 `(内半径, 高さ)` から、薄い**円板を積み上げて**容積を概算します（円板法）。

In [ ]:
import numpy as np
# 内側の輪郭（内半径mm, 高さmm）を下から
inner = [(0,8), (20,8), (30,30), (33,68)]
r = np.array([p[0] for p in inner]); h = np.array([p[1] for p in inner])
# 各区間を円板(円柱)とみなして体積を合計： π r² × 高さ
vol_mm3 = np.sum(np.pi * ((r[:-1]+r[1:])/2)**2 * np.diff(h))
print(round(vol_mm3/1000, 1), 'ml  (1ml = 1000mm³)')

### この器を「石こう型の原型」に使う流れ
1. モデルを `.scad`→OpenSCADで **`.stl`** に書き出す
2. PLA等で3Dプリント（＝原型）
3. 原型の周りに石こうを流して **型** を取る
4. 型に泥漿を流す＝**鋳込み（スリップキャスト）**で同じ器を量産

> 焼成で約10〜13%縮むので **仕上がり÷(1−収縮率)** で一回り大きく作る（`make_bowl` は収縮逆算に対応）。

## 出力ファイルの見方・操作

SolidPython が出す `.scad` は**設計レシピ（テキスト）**なので、見るには **OpenSCAD**（無料）で開きます。

**OpenSCAD での手順**
1. [openscad.org](https://openscad.org/) からインストール
2. **File → Open** で `chawan.scad` を開く（自動で3Dプレビューが出る）
3. 操作：左ドラッグ=回転 / ホイール=ズーム / 右ドラッグ=平行移動
4. **F5**=高速プレビュー、**F6**=本レンダリング（正確な形）
5. **F7**（または **File → Export → Export as STL**）で `.stl` を書き出す

### Colab で動かした場合：まずダウンロード

Colab はクラウド上で動くので、生成された `.scad` を**自分のPCに落として**から OpenSCAD で開きます。
（ローカルのJupyterなら、ノートと同じフォルダにすでに保存されています。）

In [ ]:
# Colabなら生成した.scadをダウンロード（ローカルJupyterでは不要）
try:
    from google.colab import files
    files.download('chawan.scad')          # 必要なら他の.scadも同様に
except Exception:
    import glob
    print('ローカル環境です。次の .scad がノートと同じフォルダにあります:')
    print(glob.glob('*.scad'))

### `.stl` ファイルの見方・使い道（共通）

`.stl` は3D形状の標準フォーマット。次のように扱えます。
- **見るだけ**：ダブルクリックでOSの3Dビューア（Mac=プレビュー / Win=3Dビューア）。または無料オンラインビューア（`viewstl.com` などにファイルをドラッグ）。
- **編集**：Blender や FreeCAD に読み込んで形を調整。
- **3Dプリント**：スライサー（**Cura** / **PrusaSlicer**、無料）で `.stl` を開く → 印刷設定 → プリンタへ。

> **陶芸での使い道**：プリントした器そのものは焼けませんが、**石こう型の原型**や、ろくろ・たたら作りの**テンプレート/ゲージ**として活用できます。

## 完了ログ

このノートを終えたら、下のセルに **名前** を入れて実行してください（学習の記録用）。
記録用URL（`LOG_URL`）は配布時に設定されます。空欄のままなら記録されず、メッセージが出るだけです。

In [ ]:
# === 完了ログ ===  このノートを終えたら、NAME を入れて実行してください。
import datetime
try:
    import requests
except ImportError:
    requests = None

NAME = ""      # ← あなたの名前（例: 山田）
LOG_URL = ""   # ← 記録用URL（配布時に設定。空なら記録せず表示のみ）
NOTEBOOK = "07_ホビー_陶芸3D/03_SolidPythonで器をつくる"

if NAME and LOG_URL and requests:
    try:
        requests.post(LOG_URL, json={"name": NAME, "notebook": NOTEBOOK,
                      "time": datetime.datetime.now().isoformat(timespec="seconds")}, timeout=10)
        print(f"記録しました: {NAME} / {NOTEBOOK}")
    except Exception as e:
        print("記録に失敗しました（URL/ネットワークを確認）:", e)
else:
    print(f"{NOTEBOOK}: NAME と LOG_URL を設定すると、学習の完了が記録されます。")